In [ ]:
import numpy as np


In [9]:
class CollaborativeFiltering:
    def __init__(self, Y, R, num_features, alpha=0.01, lambda_reg=0.1):
        self.Y = Y
        self.R = R
        self.num_movies, self.num_users = Y.shape
        self.num_features = num_features
        self.alpha = alpha
        self.lambda_reg = lambda_reg
        
        self.X = np.random.randn(self.num_movies, num_features)
        self.W = np.random.randn(self.num_users, num_features)
        
        self.Ynorm, self.Ymean = self.mean_normalize(Y, R)

    def mean_normalize(self, Y, R):
        Ymean = np.zeros(Y.shape[0])
        Ynorm = np.zeros(Y.shape)
        for i in range(Y.shape[0]):
            idx = R[i, :] == 1
            if np.any(idx):
                Ymean[i] = np.mean(Y[i, idx])
                Ynorm[i, idx] = Y[i, idx] - Ymean[i]
        return Ynorm, Ymean

    def compute_cost(self):
        predictions = self.X @ self.W.T
        error = (predictions - self.Ynorm) * self.R
        cost = 0.5 * np.sum(np.square(error))
        cost += 0.5 * self.lambda_reg * (np.sum(np.square(self.X)) + np.sum(np.square(self.W)))
        return cost

    def compute_gradients(self):
        predictions = self.X @ self.W.T
        error = (predictions - self.Ynorm) * self.R
        X_grad = error @ self.W + self.lambda_reg * self.X
        W_grad = error.T @ self.X + self.lambda_reg * self.W
        return X_grad, W_grad

    def train(self, num_iters=1000, verbose=True):
        for i in range(num_iters):
            X_grad, W_grad = self.compute_gradients()
            self.X -= self.alpha * X_grad
            self.W -= self.alpha * W_grad
            if verbose and i % 100 == 0:
                print(f"Iteration {i}: cost = {self.compute_cost():.4f}")

    def predict(self):
        return self.X @ self.W.T + self.Ymean[:, np.newaxis]


# --- Create Input Data for 4 users, 5 movies ---
Y = np.array([
    [5, 4, 0, 0],
    [3, 0, 0, 0],
    [4, 0, 0, 0],
    [3, 0, 0, 0],
    [0, 0, 5, 4],
])
R = (Y != 0).astype(int)
num_features = 3

# Train model
cf = CollaborativeFiltering(Y, R, num_features=num_features, alpha=0.01, lambda_reg=0.1)
cf.train(num_iters=1200)

# Predict ratings
pred = cf.predict()
print("\nPredicted Ratings:\n", np.round(pred))

Iteration 0: cost = 7.0117
Iteration 100: cost = 0.9804
Iteration 200: cost = 0.8062
Iteration 300: cost = 0.6685
Iteration 400: cost = 0.5571
Iteration 500: cost = 0.4672
Iteration 600: cost = 0.3951
Iteration 700: cost = 0.3378
Iteration 800: cost = 0.2924
Iteration 900: cost = 0.2568
Iteration 1000: cost = 0.2290
Iteration 1100: cost = 0.2075

Predicted Ratings:
 [[5. 4. 4. 5.]
 [3. 3. 3. 3.]
 [4. 4. 4. 4.]
 [3. 3. 3. 3.]
 [4. 5. 5. 4.]]
